In [ ]:
import copy
import pathlib

from laserchicken import register_new_feature_extractor
from laserchicken.feature_extractor.band_ratio_feature_extractor import BandRatioFeatureExtractor

from laserfarm import DataProcessing, MacroPipeline

# Macro-Pipeline AHN Workflow - Feature Extraction (All Points)

## Set input/output paths

In [ ]:
root_path = pathlib.Path("/project/lidarac/Share/users/fnattino")

# input path (retiled files)
input_path = root_path / "normalized"

# output path (targets files)
output_path = root_path / "targets_all"

In [ ]:
tiles = list(input_path.glob("tile_*_*.laz"))
print("Extract features for {} tiles".format(len(tiles)))

## Setup Cluster

Setup Dask cluster used for the macro-pipeline calculation.

In [ ]:
from dask.distributed import Client

client = Client("tcp://10.0.0.52:33961")
client

## Feature Extraction

We extract features for all points available.

In [ ]:
# list of features
features = [
    "pulse_penetration_ratio",
    "point_density"
]

In [ ]:
# details of the retiling schema
grid = {
    "min_x": -113107.81,
    "max_x": 398892.19,
    "min_y": 214783.87,
    "max_y": 726783.87,
    "n_tiles_side": 512
}

# target mesh size
tile_mesh_size = 10

In [ ]:
# setup input dictionary to configure the feature extraction pipeline
feature_extraction_input_all = {
    "setup_local_fs": {
        "input_folder": input_path.as_posix(),
        "output_folder": output_path.as_posix(),
    },
    "load": {"attributes": ["raw_classification"]},
    "generate_targets": {
        "tile_mesh_size" : tile_mesh_size,
        "validate" : True,
        # solves numerical issues for tiles that have points on the edge
        "validate_precision": 0.001,
        **grid
    },
    "extract_features": {
        "feature_names": features,
        "volume_type": "cell",
        "volume_size": tile_mesh_size
    },
    "export_targets": {
        "attributes": features,
        "multi_band_files": False,
        "overwrite": True
    },
    "clear_cache" : {},
}

In [ ]:
macro = MacroPipeline()

In [ ]:
# extract the tile indices from the tile names
tile_indices = [
    [int(el) for el in tile.stem.split("_")[1:]]
    for tile in tiles
]

In [ ]:
# add pipeline list to macro-pipeline object and set the corresponding labels
macro.tasks = [
    DataProcessing(t.name, tile_index=idx).config(feature_extraction_input_all)
    for t, idx in zip(tiles, tile_indices)
]

macro.set_labels([tile.stem for tile in tiles])

In [ ]:
macro.setup_cluster(cluster=client.scheduler.address)

In [ ]:
# run!
macro.run()

In [ ]:
# save outcome results
macro.print_outcome(to_file="feature_extraction_all.out")

In [ ]:
assert not macro.get_failed_pipelines(), "Some of the tasks have failed!"

## Terminate cluster

In [ ]:
# client.close()
# macro.shutdown()